# SkillsAgent — Colab runner with real TSFM (T4 GPU)

This notebook runs the **SkillsAgent ablation study** on a Colab GPU so `granite-tsfm` (forecasting + anomaly detection) executes for real instead of falling back to the mock.

**Before you start**

1. `Runtime → Change runtime type → GPU (T4)`
2. Put your project folder in Google Drive at e.g. `MyDrive/HPML/project/` containing `SkillsAgent/`, `AssetOpsBench/`, and `main.json`.
3. Run cells top-to-bottom.

**What this notebook does**

- Git-clones AssetOpsBench from GitHub + rsyncs SkillsAgent & WO sample CSVs from Drive (~1 min total)
- Installs `granite-tsfm` + SkillsAgent requirements
- Extracts per-asset CSVs from `main.json` once (→ `data/chillers/`)
- Writes a Colab `.env` (no CouchDB; uses `IOT_CSV_DIR` instead)
- Smoke-tests real TSFM on the T4
- Runs `eval_runner` on the representative subset and downloads results

In [1]:
import os, subprocess

assert subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0, \
    "GPU not attached — Runtime → Change runtime type → T4 GPU"

print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)

GPU 0: Tesla T4 (UUID: GPU-e0f282a1-7926-013b-b297-c89d5e2c4e68)



## 1. Fetch code to local disk

Drive → Colab FUSE is painfully slow for tens of thousands of small files (`.venv`, `.git` pack objects, cached artifacts), so we **git-clone AssetOpsBench from GitHub** (~30 s) and only rsync the small bits that aren't in the upstream repo:

- `AssetOpsBench/src/tmp/assetopsbench/sample_data/` — WO CSVs (~8 MB) needed for `USE_WO_SUBPROCESS`
- `SkillsAgent/` — your own code, with aggressive excludes

Total setup: ~1 minute instead of ~3 hours. If you have **local edits** to AssetOpsBench, set `ASSETOPS_REPO` below to your fork, or revert to the old rsync path.

In [2]:
!pip install -qU deepagents

# !pip install -qU "git+https://github.com/langchain-ai/deepagents.git"

In [3]:
!pip install -qU langchain-ibm ibm-watsonx-ai langchain-google-genai langchain-anthropic langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 3.1 MB/s eta 0:00:00


In [22]:
import os, subprocess, time
from google.colab import drive

drive.mount("/content/drive")

DRIVE_PROJECT = "/content/drive/MyDrive/HPML/project"
LOCAL_ROOT    = "/content/work"
ASSETOPS_REPO = "https://github.com/IBM/AssetOpsBench.git"
ASSETOPS_REF  = "main"

assert os.path.isdir(DRIVE_PROJECT), f"not found: {DRIVE_PROJECT}"
os.makedirs(LOCAL_ROOT, exist_ok=True)

aob_dir = f"{LOCAL_ROOT}/AssetOpsBench"

# 1a. AssetOpsBench: shallow git clone (seconds) instead of Drive rsync (hours).
#     If a prior attempt left a non-empty dir without `.git`, git clone exits 128
#     ("destination path already exists and is not an empty directory"); we wipe
#     partials and surface stderr so any real error is actually visible.
t0 = time.time()
if os.path.isdir(aob_dir) and not os.path.isdir(f"{aob_dir}/.git"):
    print(f"  wiping partial/empty {aob_dir}")
    subprocess.check_call(["rm", "-rf", aob_dir])

if not os.path.isdir(f"{aob_dir}/.git"):
    r = subprocess.run(
        ["git", "clone", "--depth=1", "--branch", ASSETOPS_REF,
         ASSETOPS_REPO, aob_dir],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print("--- git stdout ---\n" + (r.stdout or ""))
        print("--- git stderr ---\n" + (r.stderr or ""))
        raise RuntimeError(f"git clone failed (exit {r.returncode})")
    print(f"  AssetOpsBench cloned in {time.time()-t0:.1f}s")
else:
    print(f"  AssetOpsBench already at {aob_dir} — skipping clone")

# 1b. WO sample CSVs live at src/tmp/assetopsbench/sample_data/ — NOT in the
# upstream git repo, so rsync them (~8 MB) from Drive. Needed by FMSR
# (failure_codes, primary_failure_codes) and USE_WO_SUBPROCESS.
t0 = time.time()
drive_tmp = f"{DRIVE_PROJECT}/AssetOpsBench/src/tmp"
local_tmp = f"{aob_dir}/src/tmp"
if os.path.isdir(drive_tmp):
    os.makedirs(local_tmp, exist_ok=True)
    subprocess.check_call(["rsync", "-a", f"{drive_tmp}/", f"{local_tmp}/"])
    print(f"  WO sample_data rsynced in {time.time()-t0:.1f}s")
else:
    print(f"  SKIP src/tmp: not found at {drive_tmp} — WO tool will fall back to mock")

# 1c. SkillsAgent: small code-only rsync from Drive. Aggressive excludes skip
# OneDrive/MacOS junk and prior eval outputs (step 7 copies those back to Drive).
t0 = time.time()
subprocess.check_call([
    "rsync", "-a",
    "--exclude=.git", "--exclude=__pycache__", "--exclude=.pytest_cache",
    "--exclude=.venv", "--exclude=.DS_Store", "--exclude=eval_results",
    f"{DRIVE_PROJECT}/SkillsAgent/", f"{LOCAL_ROOT}/SkillsAgent/",
])
print(f"  SkillsAgent rsynced in {time.time()-t0:.1f}s")

SKILLS    = f"{LOCAL_ROOT}/SkillsAgent"
ASSETOPS  = f"{aob_dir}/src"
MAIN_JSON = f"{DRIVE_PROJECT}/main.json"

print()
print("SKILLS   ", SKILLS)
print("ASSETOPS ", ASSETOPS)
print("MAIN_JSON", MAIN_JSON, "(exists)" if os.path.isfile(MAIN_JSON) else "(MISSING)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  AssetOpsBench already at /content/work/AssetOpsBench — skipping clone
  WO sample_data rsynced in 0.1s
  SkillsAgent rsynced in 0.5s

SKILLS    /content/work/SkillsAgent
ASSETOPS  /content/work/AssetOpsBench/src
MAIN_JSON /content/drive/MyDrive/HPML/project/main.json (exists)


## 2. Install dependencies

Two places deps need to land, because TSFM runs via `uv run` inside a separate venv:

1. **Colab system Python** — SkillsAgent's own deps (fast).
2. **`AssetOpsBench/.venv`** — `torch`, `transformers`, and `granite-tsfm` (the big one). Upstream `pyproject.toml` explicitly says *"tsfm_public must be installed separately"* and only lists `torch`/`transformers` in the optional `[tsfm]` group, so we do both.

First run: ~3–5 min (mostly pulling CUDA torch wheel). Cached on subsequent runs in the same session.

In [26]:
import subprocess, sys, time

# 2a. SkillsAgent's own Python deps → Colab system Python.
print("→ SkillsAgent requirements...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                       "-r", f"{SKILLS}/requirements.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "ijson"])

# 2b. AssetOpsBench venv gets torch + transformers + granite-tsfm so the
#     `uv run python -c ...` TSFM subprocess can import tsfm_public.
print("→ uv (drives AssetOpsBench venv)...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "uv"])

print("→ uv sync --group tsfm  (torch CUDA + transformers, first time is slow)...")
t0 = time.time()
subprocess.check_call(["uv", "sync", "--group", "tsfm"], cwd=aob_dir)
print(f"   uv sync done in {time.time()-t0:.1f}s")

venv_py = f"{aob_dir}/.venv/bin/python"
assert os.path.isfile(venv_py), f"venv python missing: {venv_py} — did uv sync succeed?"

print("→ granite-tsfm into AssetOpsBench/.venv (explicit --python target)...")
t0 = time.time()
# --python pins the install to THIS venv; otherwise uv's resolution can land
# the package somewhere else (cache-only, or Colab system) and tsfm_public
# stays unimportable from `uv run`.
subprocess.check_call([
    "uv", "pip", "install", "--python", venv_py,
    "granite-tsfm @ git+https://github.com/ibm-granite/granite-tsfm.git",
])
print(f"   granite-tsfm installed in {time.time()-t0:.1f}s")

# transformers 4.57 pins huggingface-hub <1.0, but resolution sometimes pulls
# hf-hub 1.x (Colab system env + granite-tsfm transitive). Force back to <1.0.
print("→ pinning huggingface-hub<1.0 (transformers 4.57 constraint)...")
subprocess.check_call([
    "uv", "pip", "install", "--python", venv_py,
    "huggingface-hub<1.0",
])

# Show that the four critical packages actually landed in this venv.
r = subprocess.run(
    ["uv", "pip", "list", "--python", venv_py],
    capture_output=True, text=True,
)
hits = [ln for ln in r.stdout.splitlines()
        if ln.lower().startswith(("granite-tsfm", "torch ", "transformers ",
                                  "huggingface-hub"))]
print("venv packages:\n  " + "\n  ".join(hits) if hits else "  (none matched)")

# Final sanity check via `uv run` — mirrors how tools.py calls TSFM.
probe = (
    "import torch, tsfm_public, sys; "
    "print('torch', torch.__version__, 'CUDA?', torch.cuda.is_available(), "
    "'device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU', "
    "'| tsfm_public OK @', tsfm_public.__file__)"
)
r = subprocess.run(["uv", "run", "--no-sync", "python", "-c", probe],
                   cwd=aob_dir, capture_output=True, text=True, timeout=180)
print("--- venv sanity:", r.stdout.strip())
if r.returncode != 0:
    print("--- stderr:", r.stderr.strip())
    raise RuntimeError("AssetOpsBench venv missing deps")

→ SkillsAgent requirements...
→ uv (drives AssetOpsBench venv)...
→ uv sync --group tsfm  (torch CUDA + transformers, first time is slow)...
   uv sync done in 0.6s
→ granite-tsfm into AssetOpsBench/.venv (explicit --python target)...
   granite-tsfm installed in 1.2s
→ pinning huggingface-hub<1.0 (transformers 4.57 constraint)...
venv packages:
  granite-tsfm                       0.3.6
  huggingface-hub                    0.36.2
  torch                              2.10.0
  transformers                       4.57.6
--- venv sanity: torch 2.10.0+cu128 CUDA? True device: Tesla T4 | tsfm_public OK @ /content/work/AssetOpsBench/.venv/lib/python3.12/site-packages/tsfm_public/__init__.py


## 3. Extract per-asset CSVs from `main.json`

One-shot: streams the 1.16 GB export and writes `chiller_6.csv` / `chiller_9.csv` (~1–2 min). `USE_IOT_SUBPROCESS` will be off, so `get_sensor_data` reads these.

In [27]:
CSV_DIR = f"{SKILLS}/data/chillers"
if not os.path.isfile(f"{CSV_DIR}/chiller_6.csv"):
    !python {SKILLS}/scripts/extract_main_json.py \
        --input {MAIN_JSON} --outdir {CSV_DIR} \
        --assets 'Chiller 6,Chiller 9' --max-rows 2000
else:
    print("CSVs already extracted — skipping")

!wc -l {CSV_DIR}/*.csv

CSVs already extracted — skipping
  1501 /content/work/SkillsAgent/data/chillers/chiller_6.csv
  1501 /content/work/SkillsAgent/data/chillers/chiller_9.csv
  3002 total


## 4. Write Colab `.env`

We skip CouchDB entirely. `IOT_CSV_DIR` drives `get_sensor_data`. `PATH_TO_MODELS_DIR` tells AssetOpsBench where the pretrained TTM checkpoints live.

**Paste your own API keys below** (watsonx is the default LLM provider).

In [28]:
from getpass import getpass

WATSONX_API_KEY    = os.environ.get("WATSONX_API_KEY")    or getpass("WATSONX_API_KEY: ")
WATSONX_PROJECT_ID = os.environ.get("WATSONX_PROJECT_ID") or input("WATSONX_PROJECT_ID: ")
GROQ_API_KEY       = os.environ.get("GROQ_API_KEY")       or ""
GEMINI_API_KEY     = os.environ.get("GEMINI_API_KEY")     or ""

env = f"""\
LLM_PROVIDER=watsonx
WATSONX_API_KEY={WATSONX_API_KEY}
WATSONX_PROJECT_ID={WATSONX_PROJECT_ID}
WATSONX_URL=https://us-south.ml.cloud.ibm.com
WATSONX_MODEL_ID=meta-llama/llama-4-maverick-17b-128e-instruct-fp8,meta-llama/llama-3-3-70b-instruct,mistral-large-2512
WATSONX_MAX_RETRIES=2
GROQ_API_KEY={GROQ_API_KEY}
GEMINI_API_KEY={GEMINI_API_KEY}
GEMINI_MODEL=gemini-2.5-flash

ASSETOPS={ASSETOPS}
PATH_TO_MODELS_DIR={ASSETOPS}/servers/tsfm/artifacts/tsfm_models
IOT_CSV_DIR={CSV_DIR}
COUCHDB_EXPORT_PATH={MAIN_JSON}

USE_IOT_SUBPROCESS=0
USE_TSFM_SUBPROCESS=1
USE_WO_SUBPROCESS=1
USE_FMSR_SUBPROCESS=1
WO_CSV_DIR={ASSETOPS}/tmp/assetopsbench/sample_data
TSFM_SUBPROCESS_TIMEOUT=600
TSFM_MIN_CONTEXT_ROWS=96
"""
open(f"{SKILLS}/.env", "w").write(env)
print("wrote", f"{SKILLS}/.env")
print(env)

wrote /content/work/SkillsAgent/.env
LLM_PROVIDER=watsonx
WATSONX_API_KEY=8XywHLJHicXCTTwC1pkz7fVXrZTBKUoziGixx9Vz7ryj
WATSONX_PROJECT_ID=178e05b2-3352-4f21-8388-572e6b13d65d
WATSONX_URL=https://us-south.ml.cloud.ibm.com
WATSONX_MODEL_ID=meta-llama/llama-4-maverick-17b-128e-instruct-fp8,meta-llama/llama-3-3-70b-instruct,mistral-large-2512
WATSONX_MAX_RETRIES=2
GROQ_API_KEY=
GEMINI_API_KEY=
GEMINI_MODEL=gemini-2.5-flash

ASSETOPS=/content/work/AssetOpsBench/src
PATH_TO_MODELS_DIR=/content/work/AssetOpsBench/src/servers/tsfm/artifacts/tsfm_models
IOT_CSV_DIR=/content/work/SkillsAgent/data/chillers
COUCHDB_EXPORT_PATH=/content/drive/MyDrive/HPML/project/main.json

USE_IOT_SUBPROCESS=0
USE_TSFM_SUBPROCESS=1
USE_WO_SUBPROCESS=1
USE_FMSR_SUBPROCESS=1
WO_CSV_DIR=/content/work/AssetOpsBench/src/tmp/assetopsbench/sample_data
TSFM_SUBPROCESS_TIMEOUT=600
TSFM_MIN_CONTEXT_ROWS=96



## 5. Smoke-test real TSFM on the T4

First run compiles the model (~30–60 s on T4); subsequent calls are <2 s. A successful run prints `forecast source: tsfm_subprocess`.

In [29]:
import sys, time
sys.path.insert(0, SKILLS)
os.chdir(SKILLS)

for m in list(sys.modules):
    if m in ("tools", "knowledge", "agent", "skills", "confidence_evaluator"):
        del sys.modules[m]

from dotenv import load_dotenv
load_dotenv(f"{SKILLS}/.env", override=True)
from tools import get_sensor_data, forecast_sensor, deep_tsfm_refine_anomalies, detect_anomaly

d = get_sensor_data("Chiller 6", lookback_days=14)
print("IoT source:", d["source"], "rows:", d.get("iot_total_observations"))

t0 = time.time()
f = forecast_sensor("Chiller 6", "flow_rate_GPM", horizon_days=7, sensor_data=d)
print(f"forecast source={f['source']}  elapsed={time.time()-t0:.1f}s  head={f.get('forecasted', [])[:3]}")

basic = detect_anomaly(d)
t0 = time.time()
deep = deep_tsfm_refine_anomalies("Chiller 6", d, basic)
print(
    f"deep tsad  refined={deep.get('deep_tsfm_refined')}"
    f"  tsad_records={deep.get('tsfm_integrated_tsad_records', 0)}"
    f"  anomalies_detected={deep.get('anomalies_detected')}"
    f"  severity={deep.get('severity')}"
    f"  n_details={len(deep.get('anomaly_details', []))}"
    f"  elapsed={time.time()-t0:.1f}s"
)

IoT source: iot_csv rows: 1500
forecast source=tsfm_subprocess  elapsed=14.4s  head=[0.6911484003067017, 0.5683184862136841, 0.622434675693512]
deep tsad  refined=True  tsad_records=45  anomalies_detected=True  severity=high  n_details=15  elapsed=16.5s


## 5b. [Archived] Deep TSAD bug reproducer — expected to fail with `KeyError('segment_id')`

**If cell 5 printed `forecast source=tsfm_subprocess` and `tsad_records>0`, skip this cell.**
It is only kept for regression debugging.

This cell intentionally calls `run_integrated_tsad(..., id_columns=["segment_id"], ...)` without
the `column_specifiers` sanitization that `tools.py` applies in production. It reproduces the
original bug we saw in `colab_20260422_0839`, which triggered the two fixes now in
`tools.py::_tsfm_integrated_tsad_subprocess`:

1. Do not pass `id_columns=["segment_id"]` — `forecasting.py` auto-injects `segment_id` internally.
2. Strip `id_columns` from `column_specifiers` before the dataset config is built, otherwise
   `ForecastDFDataset()` raises `got multiple values for keyword argument 'id_columns'`.

Cell 5d below re-runs the same inputs with those two fixes applied and should succeed end-to-end.

In [9]:
import csv, json, os, subprocess, tempfile, sys
sys.path.insert(0, SKILLS)
os.chdir(SKILLS)
for m in list(sys.modules):
    if m in ("tools",):
        del sys.modules[m]
from tools import get_sensor_data, detect_anomaly, _write_tsfm_series_csv, _deep_tsfm_primary_column, _assetops_repo_root

d = get_sensor_data("Chiller 6", lookback_days=14)
basic = detect_anomaly(d)
readings = d["readings"]
col = _deep_tsfm_primary_column(basic, readings)
print("chosen column:", col, "| rows:", len(readings.get(col, [])))

fd, csv_path = tempfile.mkstemp(suffix=".csv")
os.close(fd)
ok = _write_tsfm_series_csv(readings, col, csv_path, min_rows=96)
print("csv written:", ok, "->", csv_path)

with open(csv_path) as fh:
    lines = fh.readlines()
print("csv rows:", len(lines), "| head:", lines[:3])

script = f'''
import sys, json
sys.path.insert(0, "src")
from servers.tsfm.main import run_integrated_tsad
r = run_integrated_tsad(
    dataset_path={json.dumps(csv_path)},
    timestamp_column="timestamp",
    target_columns=[{json.dumps(col)}],
    model_checkpoint="ttm_96_28",
    id_columns=["segment_id"],
    frequency_sampling="oov",
    autoregressive_modeling=True,
)
d = r.model_dump() if hasattr(r, "model_dump") else r.dict()
print(json.dumps({{"keys": list(d.keys()), "error": d.get("error"), "results_file": d.get("results_file")}}))
'''
proc = subprocess.run(
    ["uv", "run", "--no-sync", "python", "-c", script],
    cwd=str(_assetops_repo_root()),
    capture_output=True, text=True, timeout=300, env=os.environ.copy(),
)
print("\n--- returncode:", proc.returncode)
print("--- stdout:", proc.stdout[-1500:])
print("--- stderr:", proc.stderr[-1500:])

rf = None
for line in reversed((proc.stdout or "").splitlines()):
    try:
        j = json.loads(line)
        rf = j.get("results_file")
        break
    except Exception:
        continue
if rf and os.path.isfile(rf):
    with open(rf) as fh:
        rows = list(csv.DictReader(fh))
    print(f"\nresults_file rows: {len(rows)}")
    print("columns:", rows[0].keys() if rows else None)
    labs = [r.get("anomaly_label") for r in rows]
    print("first labels:", labs[:10], "| non-zero count:", sum(1 for l in labs if str(l) not in ("", "0", "0.0", "False", "false")))

## 5c. Track down the `'segment_id'` KeyError

`run_integrated_tsad` returned `"No TSAD results produced"` because internal code raised `KeyError('segment_id')`. Inspect the signature + source to find the right parameter to pass.

In [ ]:
import subprocess, json, os
# 1) Show signature + source of run_integrated_tsad
probe = """
import inspect, sys
sys.path.insert(0, 'src')
from servers.tsfm.main import run_integrated_tsad
sig = inspect.signature(run_integrated_tsad)
print('SIGNATURE:', sig)
print()
src = inspect.getsource(run_integrated_tsad)
print(src[:4000])
"""
p = subprocess.run(['uv','run','--no-sync','python','-c',probe], cwd=ASSETOPS.rsplit('/src',1)[0], capture_output=True, text=True, timeout=120, env=os.environ.copy())
print(p.stdout)
print('---stderr---', p.stderr[-500:])

In [ ]:
import subprocess, os
# 2) Grep the server code for 'segment_id' + related defaults so we know what's expected
probe = r'''
grep -RInE "segment_id|id_columns" src/servers/tsfm/ 2>/dev/null | head -50
echo ---
grep -RInE "segment_id" .venv/lib/python*/site-packages/tsfm_public/ 2>/dev/null | head -20
'''
p = subprocess.run(['bash','-lc',probe], cwd=ASSETOPS.rsplit('/src',1)[0], capture_output=True, text=True, timeout=30)
print(p.stdout)
print('---stderr---', p.stderr[-500:])

In [ ]:
# 5d. Verify the anomaly.py `id_columns` duplicate-kwarg fix end-to-end
# (tools.py applies the same patch in its real subprocess too)
import subprocess, os, json, tempfile, sys
for m in list(sys.modules):
    if m == 'tools': del sys.modules[m]
sys.path.insert(0, SKILLS); os.chdir(SKILLS)
from tools import get_sensor_data, detect_anomaly, _write_tsfm_series_csv, _deep_tsfm_primary_column, _assetops_repo_root

d = get_sensor_data('Chiller 6', lookback_days=14)
basic = detect_anomaly(d)
col = _deep_tsfm_primary_column(basic, d['readings'])
fd, csv_path = tempfile.mkstemp(suffix='.csv'); os.close(fd)
_write_tsfm_series_csv(d['readings'], col, csv_path, min_rows=96)
print('csv:', csv_path, '| col:', col)

script = f'''
import sys, json, traceback
sys.path.insert(0, "src")

# NumPy 2.0 shim: AssetOpsBench TSAD uses np.infty (removed in NumPy 2.0).
import numpy as _np
if not hasattr(_np, "infty"):
    _np.infty = _np.inf
if not hasattr(_np, "Inf"):
    _np.Inf = _np.inf

# Monkeypatch: anomaly._get_tsfm_dataloaders spreads column_specifiers AND passes
# id_columns as a kwarg; forecasting.py injects id_columns into column_specifiers,
# causing "got multiple values for keyword argument 'id_columns'". Strip it.
from servers.tsfm import anomaly as A
_orig = A._get_tsfm_dataloaders
def _patched(df_dataframe, model_config, dataset_config_dictionary, scaling=False):
    cs = dataset_config_dictionary.get("column_specifiers", {{}})
    if isinstance(cs, dict) and "id_columns" in cs:
        cfg = dict(dataset_config_dictionary)
        cfg["column_specifiers"] = {{k: v for k, v in cs.items() if k != "id_columns"}}
        dataset_config_dictionary = cfg
    return _orig(df_dataframe, model_config, dataset_config_dictionary, scaling=scaling)
A._get_tsfm_dataloaders = _patched
from servers.tsfm import main as M
if hasattr(M, "_get_tsfm_dataloaders"):
    M._get_tsfm_dataloaders = _patched

from servers.tsfm.main import run_integrated_tsad
try:
    r = run_integrated_tsad(
        dataset_path={json.dumps(csv_path)},
        timestamp_column="timestamp",
        target_columns=[{json.dumps(col)}],
        model_checkpoint="ttm_96_28",
        frequency_sampling="oov",
        autoregressive_modeling=True,
    )
    dd = r.model_dump() if hasattr(r, "model_dump") else r.dict()
    print("RESULT keys:", list(dd.keys()), "error:", dd.get("error"))
    rf = dd.get("results_file")
    print("results_file:", rf)
    if rf:
        import csv as _csv
        with open(rf) as fh:
            rows = list(_csv.DictReader(fh))
        print("rows:", len(rows), "cols:", list(rows[0].keys()) if rows else None)
        labels = [r.get("anomaly_label") for r in rows]
        nz = sum(1 for l in labels if str(l) not in ("", "0", "0.0", "False", "false"))
        print("non-zero anomaly_label count:", nz)
except Exception:
    traceback.print_exc()
'''
p = subprocess.run(['uv','run','--no-sync','python','-c',script], cwd=str(_assetops_repo_root()),
                   capture_output=True, text=True, timeout=300, env=os.environ.copy())
print('--- stdout ---')
print(p.stdout)
print('--- stderr tail ---')
print(p.stderr[-1000:])


## 6a. Calibrate per-skill wall-clock latency on this GPU

Runs each skill 3× on a representative task and writes `skill_costs.json`. The evaluator picks this up automatically so Condition~E's `cost_budget` reflects T4 latencies, not Mac-CPU priors.

In [30]:
!cd {SKILLS} && python scripts/calibrate_costs.py --runs 3 --output skill_costs.json

import json
print(json.dumps(json.load(open(f'{SKILLS}/skill_costs.json')), indent=2))

Calibrating 7 skills × 3 runs (task: 'Diagnose anomalies on Chiller 6 and generate a work order')

  data_retrieval           median=  0.011s   runs=['0.017', '0.011', '0.010']
  metadata_retrieval       median=  2.659s   runs=['3.763', '2.519', '2.659']
  anomaly_detection        median=  6.621s   runs=['6.621', '5.978', '7.635']
  root_cause_analysis      median= 23.874s   runs=['25.645', '23.610', '23.874']
  validate_failure         median=  0.000s   runs=['0.000', '0.000', '0.000']
  forecasting              median= 20.682s   runs=['20.682', '21.028', '16.666']
  work_order_generation    median=  0.845s   runs=['0.845', '0.824', '1.020']

  __deep_tsfm__            median= 16.331s

wrote skill_costs.json  (54.69s total per full plan)
{
  "data_retrieval": 0.0106,
  "metadata_retrieval": 2.6589,
  "anomaly_detection": 6.6209,
  "root_cause_analysis": 23.8741,
  "validate_failure": 0.0,
  "forecasting": 20.682,
  "work_order_generation": 0.8446,
  "__deep_tsfm__": 16.3306
}


## 6b. Deep Agent Setup

In [31]:
import deepagents
print("deepagents OK")

deepagents OK


In [32]:
!cd /content/work/SkillsAgent && cat .env | grep -E "LLM_PROVIDER|WATSONX|ASSETOPS|IOT_CSV_DIR|USE_TSFM|PATH_TO_MODELS"

LLM_PROVIDER=watsonx
WATSONX_API_KEY=8XywHLJHicXCTTwC1pkz7fVXrZTBKUoziGixx9Vz7ryj
WATSONX_PROJECT_ID=178e05b2-3352-4f21-8388-572e6b13d65d
WATSONX_URL=https://us-south.ml.cloud.ibm.com
WATSONX_MODEL_ID=meta-llama/llama-4-maverick-17b-128e-instruct-fp8,meta-llama/llama-3-3-70b-instruct,mistral-large-2512
WATSONX_MAX_RETRIES=2
ASSETOPS=/content/work/AssetOpsBench/src
PATH_TO_MODELS_DIR=/content/work/AssetOpsBench/src/servers/tsfm/artifacts/tsfm_models
IOT_CSV_DIR=/content/work/SkillsAgent/data/chillers
USE_TSFM_SUBPROCESS=1


In [33]:
import os, sys
from dotenv import load_dotenv

SKILLS = "/content/work/SkillsAgent"
load_dotenv(f"{SKILLS}/.env", override=True)

print("cwd:", os.getcwd())
print("WATSONX_API_KEY present:", bool(os.getenv("WATSONX_API_KEY")))
print("WATSONX_PROJECT_ID present:", bool(os.getenv("WATSONX_PROJECT_ID")))
print("LLM_PROVIDER:", os.getenv("LLM_PROVIDER"))

import langchain_ibm
print("langchain_ibm OK")

from deep_agent import _init_model
m = _init_model()
print("DeepAgent model:", type(m))

cwd: /content/work/SkillsAgent
WATSONX_API_KEY present: True
WATSONX_PROJECT_ID present: True
LLM_PROVIDER: watsonx
langchain_ibm OK
DeepAgent model: <class 'langchain_ibm.chat_models.ChatWatsonx'>


## 6. Run eval on the representative subset

Uses the built-in `BUILTIN_TASK_BANK` (no HF download). To sweep more scenarios pass `--hf-limit 20`. All conditions A–E run; theta sweep included.

In [34]:
OUT = f"{SKILLS}/eval_results/colab_{time.strftime('%Y%m%d_%H%M')}"
os.makedirs(OUT, exist_ok=True)

!cd {SKILLS} && python -m eval_runner_deep_agent \
    --output-dir {OUT} \
    --trajectory-log {OUT}/trajectories.jsonl 2>&1

print("\nfiles:")
!ls -la {OUT}


────────────────────────────────────────────────────────────
Condition: A_raw_llm
────────────────────────────────────────────────────────────
  T01 [fault_diagnosis] Why is Chiller 6 behaving abnormally and do we need a w...
    ok  calls=1 deep_tsfm=False cost=0.0 lat=4.42s
  T02 [forecasting] Forecast next week's condenser water flow for Chiller 9...
    ok  calls=1 deep_tsfm=False cost=0.0 lat=4.034s
  T03 [anomaly_detection] Was there any abnormal behavior in Chiller 9 over the p...
    ok  calls=1 deep_tsfm=False cost=0.0 lat=3.678s
  T04 [metadata] What sensors are available for Chiller 6, and what do t...
    ok  calls=1 deep_tsfm=False cost=0.0 lat=3.837s
  T05 [fault_diagnosis] Chiller 9 vibration has been rising — should we schedul...
    ok  calls=1 deep_tsfm=False cost=0.0 lat=3.766s
  T06 [forecasting] Predict COP for Chiller 6 next month and flag if mainte...
    ok  calls=1 deep_tsfm=False cost=0.0 lat=3.682s
  T07 [fault_diagnosis] Chiller 6 refrigerant pressure is dr

## 7. Copy results back to Drive

So you keep them after the runtime is recycled.

In [35]:
DRIVE_OUT = f"{DRIVE_PROJECT}/SkillsAgent/eval_results/{os.path.basename(OUT)}"
os.makedirs(DRIVE_OUT, exist_ok=True)
subprocess.check_call(["rsync", "-a", f"{OUT}/", f"{DRIVE_OUT}/"])
print("copied →", DRIVE_OUT)

copied → /content/drive/MyDrive/HPML/project/SkillsAgent/eval_results/colab_20260503_1831
